# 🍳 Optimizer Cookbook — Chapter 02: Adagrad

**Previous:** `01_sgd_momentum.ipynb` — SGD + Momentum  
**Next:** `03_rmsprop.ipynb` — RMSprop

---

## The Problem with SGD (Even with Momentum)

SGD and SGD+Momentum use **one global learning rate** for every parameter.  
This is a problem because:

- **Frequent features** (common words, common pixels) get large gradients and need a *small* LR to avoid overshooting
- **Rare features** (infrequent words, sparse activations) get tiny gradients and need a *large* LR to learn at all

A single LR cannot serve both well. **Adagrad solves this** by giving each parameter its own adaptive learning rate.

---

## What is Adagrad?

**Adagrad** (Adaptive Gradient Algorithm, Duchi et al. 2011) accumulates the sum of squared past gradients per parameter and scales the LR inversely — parameters with historically large gradients get smaller steps, and vice versa.

---

## Update Rule

$$G_{t} = G_{t-1} + (\nabla_{\theta} \mathcal{L})^2$$

$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{G_t + \epsilon}} \cdot \nabla_{\theta} \mathcal{L}$$

Where:
- $G_t$ = sum of squared gradients up to time $t$ (per parameter)
- $\epsilon$ = small constant for numerical stability (default: `1e-10`)
- $\eta$ = initial learning rate (typically **0.01**)

**Key insight:** $G_t$ only ever grows — so the effective LR $\frac{\eta}{\sqrt{G_t}}$ only ever shrinks.

---

## The Fatal Flaw: Monotonically Decaying LR

Because $G_t$ is a cumulative sum that never decreases, the learning rate **keeps shrinking toward zero** — eventually the model stops learning entirely. This makes Adagrad:
- ✅ Great for **convex, sparse problems** (e.g. text classification, word embeddings)
- ❌ Bad for **deep networks trained over many epochs** (LR dies too early)

RMSprop (next chapter) fixes this by using an **exponential moving average** instead of a cumulative sum.

---

## When to Use Adagrad

| Scenario | Verdict |
|---|---|
| Sparse NLP tasks (bag-of-words, TF-IDF) | ✅ Excellent |
| Word embedding training (Word2Vec style) | ✅ Excellent |
| Convex optimization problems | ✅ Good |
| Deep CNNs trained over 20+ epochs | ❌ LR decays too aggressively |
| RNNs / LSTMs | ❌ RMSprop preferred |
| Large-scale deep learning | ❌ Adam preferred |

---

## Key Hyperparameters

| Parameter | Default | Effect |
|---|---|---|
| `lr` | 0.01 | Initial LR — will decay automatically |
| `lr_decay` | 0 | Optional additional LR decay per step |
| `eps` | 1e-10 | Numerical stability — rarely needs changing |
| `weight_decay` | 0 | L2 regularization |

---
## 0. Imports & CUDA Check

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os, time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device : {device}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')

---
## 1. Config

In [ ]:
BATCH_SIZE     = 128
NUM_EPOCHS     = 20
NUM_CLASSES    = 10
NUM_WORKERS    = 2
SEED           = 42

# ── Optimizer Config ─────────────────────────────────────────
LEARNING_RATE  = 0.01
LR_DECAY       = 0
EPS            = 1e-10
WEIGHT_DECAY   = 5e-4
OPTIMIZER_NAME = 'Adagrad'

DATA_DIR    = '../data'
RESULTS_DIR = '../results/logs'
PLOTS_DIR   = '../results/plots'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
print('Config loaded ✓')

---
## 2. Dataset — CIFAR-10

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_dataset = torchvision.datasets.CIFAR10(root=DATA_DIR, train=True,  download=True, transform=train_transform)
test_dataset  = torchvision.datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=test_transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

---
## 3. Model — SimpleCNN

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

torch.manual_seed(SEED)
model = SimpleCNN(num_classes=NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model ready. Trainable params: {total_params:,}')

---
## 4. Optimizer — Adagrad

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adagrad(
    model.parameters(),
    lr           = LEARNING_RATE,
    lr_decay     = LR_DECAY,
    eps          = EPS,
    weight_decay = WEIGHT_DECAY
)

print(f'Optimizer    : Adagrad')
print(f'LR           : {LEARNING_RATE}')
print(f'LR Decay     : {LR_DECAY}')
print(f'Eps          : {EPS}')
print(f'Weight Decay : {WEIGHT_DECAY}')
print('-' * 65)

---
## 5. Training Utilities

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, 100.0 * correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    return running_loss / total, 100.0 * correct / total

def run_training(model, train_loader, test_loader, optimizer, criterion, num_epochs, device, optimizer_name):
    history = []
    best_acc = 0.0
    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc     = evaluate(model, test_loader, criterion, device)
        elapsed = time.time() - t0
        if val_acc > best_acc: best_acc = val_acc
        history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                        'val_loss': val_loss, 'val_acc': val_acc, 'time_s': elapsed})
        print(f'Epoch [{epoch:02d}/{num_epochs}] Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | '
              f'Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | Time: {elapsed:.1f}s')
    print(f'\n✓ Best Val Accuracy: {best_acc:.2f}%')
    df = pd.DataFrame(history)
    df.to_csv(f'{RESULTS_DIR}/{optimizer_name}_log.csv', index=False)
    print(f'✓ Log saved → results/logs/{optimizer_name}_log.csv')
    return df

print('Utilities loaded ✓')

---
## 6. Train

In [ ]:
history = run_training(
    model          = model,
    train_loader   = train_loader,
    test_loader    = test_loader,
    optimizer      = optimizer,
    criterion      = criterion,
    num_epochs     = NUM_EPOCHS,
    device         = device,
    optimizer_name = OPTIMIZER_NAME
)

---
## 7. Plot Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Training Curves — {OPTIMIZER_NAME}', fontsize=14, fontweight='bold')

ax1.plot(history['epoch'], history['train_loss'], label='Train Loss', linewidth=2, color='steelblue')
ax1.plot(history['epoch'], history['val_loss'],   label='Val Loss',   linewidth=2, color='tomato', linestyle='--')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history['epoch'], history['train_acc'], label='Train Acc', linewidth=2, color='steelblue')
ax2.plot(history['epoch'], history['val_acc'],   label='Val Acc',   linewidth=2, color='tomato', linestyle='--')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.set_title('Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/{OPTIMIZER_NAME}_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Plot saved → results/plots/{OPTIMIZER_NAME}_curves.png')

---
## 8. Visualising the Dying LR Problem

This cell demonstrates Adagrad's core flaw — the effective learning rate per layer decaying toward zero over training.

We track the **effective LR** = $\frac{\eta}{\sqrt{G_t + \epsilon}}$ for the first Conv layer's weights.

In [ ]:
# Re-train a fresh model while tracking effective LR
torch.manual_seed(SEED)
track_model = SimpleCNN(num_classes=NUM_CLASSES).to(device)
track_opt   = optim.Adagrad(track_model.parameters(), lr=LEARNING_RATE, eps=EPS)
track_crit  = nn.CrossEntropyLoss()

TRACK_EPOCHS = 20
effective_lrs = []

for epoch in range(1, TRACK_EPOCHS + 1):
    track_model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        track_opt.zero_grad()
        loss = track_crit(track_model(images), labels)
        loss.backward()
        track_opt.step()
        break  # one batch per epoch is enough to show the trend

    # Extract accumulated gradient sum for first conv layer
    state = track_opt.state[list(track_model.parameters())[0]]
    if 'sum' in state:
        G = state['sum']
        eff_lr = (LEARNING_RATE / (G.sqrt() + EPS)).mean().item()
        effective_lrs.append(eff_lr)

plt.figure(figsize=(9, 4))
plt.plot(range(1, len(effective_lrs)+1), effective_lrs, color='tomato', linewidth=2, marker='o')
plt.title('Adagrad — Effective LR Decay Over Epochs (Conv1 weights)', fontsize=12, fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('Mean Effective LR')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/Adagrad_lr_decay.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Start LR: {effective_lrs[0]:.6f} → End LR: {effective_lrs[-1]:.6f}')
print(f'LR dropped by {(1 - effective_lrs[-1]/effective_lrs[0])*100:.1f}% over {TRACK_EPOCHS} epochs')

---
## 9. Cumulative Comparison vs Previous Optimizers

In [ ]:
logs = {
    'Vanilla SGD'    : 'SGD_baseline_log.csv',
    'SGD + Momentum' : 'SGD_Momentum_log.csv',
    'Adagrad'        : 'Adagrad_log.csv',
}
colors = ['gray', 'steelblue', 'darkorange']
styles = ['--', '--', '-']

plt.figure(figsize=(10, 5))
for (label, fname), color, style in zip(logs.items(), colors, styles):
    path = f'{RESULTS_DIR}/{fname}'
    if os.path.exists(path):
        df = pd.read_csv(path)
        plt.plot(df['epoch'], df['val_acc'], label=label, color=color, linestyle=style, linewidth=2)
    else:
        print(f'⚠️  Missing: {fname} — run earlier notebooks first')

plt.title('Val Accuracy — SGD vs Momentum vs Adagrad', fontsize=13, fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('Val Accuracy (%)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/comparison_up_to_adagrad.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 10. Results Summary & Analysis

In [ ]:
best_epoch = history.loc[history['val_acc'].idxmax()]

print('=' * 55)
print(f'  {OPTIMIZER_NAME} — Summary')
print('=' * 55)
print(f"  Best Val Accuracy : {best_epoch['val_acc']:.2f}%")
print(f"  Best Val Loss     : {best_epoch['val_loss']:.4f}")
print(f"  Achieved at Epoch : {int(best_epoch['epoch'])}")
print(f"  Avg Time/Epoch    : {history['time_s'].mean():.1f}s")
print('=' * 55)
print()
print('📌 Key Observations:')
print('  ✅ Fast early convergence — adaptive LR kicks in immediately')
print('  ✅ Works well on sparse features without any LR tuning')
print('  ⚠️  LR decays aggressively — model may plateau or stall mid-training')
print('  ⚠️  On dense data like CIFAR-10, SGD+Momentum usually outperforms it')
print('  ❌ Not recommended for deep nets trained over many epochs')
print()
print('📌 Real-world use cases:')
print('  → Training GloVe / Word2Vec embeddings')
print('  → Logistic regression on sparse TF-IDF features')
print('  → Convex optimization with infrequent feature updates')

---
## 11. What's Next?

Adagrad's dying LR is its Achilles heel.  
**RMSprop** (Hinton, 2012) fixes this by replacing the cumulative sum $G_t$ with an **exponential moving average** — so old gradients are forgotten and the LR stays alive.

➡️ Continue to `03_rmsprop.ipynb`